In [1]:
import pandas as pd, numpy as np

## Measured data

In [37]:
depths = [2, 10, 30, 50]

In [2]:
measured_path = r'Calibration\Profiler_modem_PFL_Step.dat' # r'Calibration\Profiler_modem_SondeHourly.dat'
measured_df = pd.read_csv(measured_path, delimiter=',', header=0, skiprows=[0, 2, 3], low_memory=False)
measured_df["TIMESTAMP"] = pd.to_datetime(measured_df["TIMESTAMP"])
measured_df["temperature"] = pd.to_numeric(measured_df["sensorParms(1)"], errors='coerce')
measured_df["depth"] = pd.to_numeric(measured_df["sensorParms(9)"], errors='coerce')
measured_df = measured_df[["TIMESTAMP", "temperature", "depth"]]
# interpolate missing values in the measured data
df_interpolated = measured_df.interpolate(method='linear', limit_direction='both')
# Get data for the year 2024
measured_2024 = df_interpolated[df_interpolated["TIMESTAMP"].dt.year == 2024].reset_index(drop=True)

In [9]:
df = (measured_2024.sort_values("TIMESTAMP").reset_index(drop=True))
new_profile = ((df["depth"] < 1.5) & (df["depth"].shift(1) > 5))
df['profile'] = new_profile.cumsum()

In [42]:
df.head(60)

,TIMESTAMP,temperature,depth,profile
0,2024-04-19 14:37:44,3.2860,0.943,0
1,2024-04-19 14:39:17,3.2780,2.091,0
2,2024-04-19 14:40:50,3.2665,3.091,0
3,2024-04-19 14:42:22,3.2550,4.090,0
4,2024-04-19 14:43:54,3.2540,5.095,0
5,2024-04-19 14:45:26,3.2500,6.022,0
6,2024-04-19 14:46:59,3.2290,7.088,0
7,2024-04-19 14:48:32,3.2390,8.032,0
8,2024-04-19 14:50:05,3.2460,9.046,0
9,2024-04-19 14:51:38,3.2540,10.072,0


In [38]:
def interpolate_profile_time(group, target_depths):
    group = group.sort_values("depth").drop_duplicates("depth").reset_index(drop=True)
    result, time = [], group["TIMESTAMP"].iloc[0]
    time_seconds = (group["TIMESTAMP"] - time).dt.total_seconds().to_numpy()
    depths = group["depth"].to_numpy()
    temperatures = group["temperature"].to_numpy()
    for target_depth in target_depths:
        if not (depths.min() <= target_depth <= depths.max()):
            continue
        # Interpolate temperature at the target depth
        temperature = np.interp(target_depth, depths, temperatures)
        elapsed_seconds = np.interp(target_depth, depths, time_seconds)
        timestamp = time + pd.to_timedelta(elapsed_seconds, unit="s").round("1s")
        row = {"TIMESTAMP": timestamp}    
        for depth in target_depths:
            row[f"temp_{depth}"] = np.nan
        row[f"temp_{target_depth}"] = temperature
        result.append(row)
    return pd.DataFrame(result)
temp_profiles = (df.groupby("profile").apply(lambda g: interpolate_profile_time(g, depths))
                 .reset_index(drop=True).sort_values("TIMESTAMP").reset_index(drop=True))

In [39]:
temp_profiles

,TIMESTAMP,temp_2,temp_10,temp_30,temp_50
0,2024-04-19 14:39:10,3.278634,NaN,NaN,NaN
1,2024-04-19 14:51:31,NaN,3.253439,NaN,NaN
2,2024-04-19 15:22:33,NaN,NaN,3.258940,NaN
3,2024-04-19 15:53:56,NaN,NaN,NaN,3.275903
4,2024-04-20 00:03:25,3.245901,NaN,NaN,NaN
...,...,...,...,...,...
1581,2024-11-14 12:16:22,NaN,6.870946,NaN,NaN
1582,2024-11-14 12:48:56,NaN,NaN,6.861921,NaN
1583,2024-11-15 12:03:24,6.845949,NaN,NaN,NaN
1584,2024-11-15 12:16:15,NaN,6.843211,NaN,NaN


In [44]:
def interpolate_profile(group, target_depths):
    group = group.sort_values("depth").drop_duplicates("depth").reset_index(drop=True)
    t_start, t_end = group["TIMESTAMP"].iloc[0], group["TIMESTAMP"].iloc[-1]
    profile_timestamp = (t_start + (t_end - t_start) / 2).round("1s")
    depths = group["depth"].to_numpy()
    temperatures = group["temperature"].to_numpy()
    result = {
        "profile_timestamp": profile_timestamp,
        "profile_start": t_start,
        "profile_end": t_end,
    }
    for target_depth in target_depths:
        if depths.min() <= target_depth <= depths.max():
            temperature = np.interp(
                target_depth,
                depths,
                temperatures
            )
            result[f"temp_{target_depth}"] = temperature
        else:
            result[f"temp_{target_depth}"] = np.nan
    return pd.Series(result)
profiles = (
    df.groupby("profile")
      .apply(lambda g: interpolate_profile(g, depths))
      .reset_index(drop=True)
      .sort_values("profile_timestamp")
      .reset_index(drop=True)
)

In [45]:
profiles

,profile_timestamp,profile_start,profile_end,temp_2,temp_10,temp_30,temp_50
0,2024-04-19 15:15:57,2024-04-19 14:37:44,2024-04-19 15:54:10,3.278634,3.253439,3.258940,3.275903
1,2024-04-20 00:41:20,2024-04-20 00:01:53,2024-04-20 01:20:48,3.245901,3.250998,3.266037,3.318433
2,2024-04-20 12:40:17,2024-04-20 12:01:52,2024-04-20 13:18:42,3.309588,3.229459,3.306000,3.372569
3,2024-04-21 00:41:18,2024-04-21 00:01:53,2024-04-21 01:20:43,3.322429,3.385510,3.398826,3.401000
4,2024-04-21 12:40:07,2024-04-21 12:01:48,2024-04-21 13:18:26,3.418110,3.304809,3.297955,3.343684
...,...,...,...,...,...,...,...
415,2024-11-13 00:41:32,2024-11-13 00:01:53,2024-11-13 01:21:11,6.899886,6.894000,6.905000,6.717498
416,2024-11-13 12:41:32,2024-11-13 12:01:54,2024-11-13 13:21:11,6.893120,6.897000,6.898938,NaN
417,2024-11-14 00:41:42,2024-11-14 00:01:54,2024-11-14 01:21:30,6.838000,6.838118,6.841003,NaN
418,2024-11-14 12:41:42,2024-11-14 12:01:54,2024-11-14 13:21:30,6.866888,6.870946,6.861921,NaN


In [ ]:
profiles["date"] = profiles["profile_timestamp"].dt.date



profiles["target_00"] = pd.to_datetime(
    profiles["date"].astype(str) + " 00:00:00"
)
profiles["target_12"] = pd.to_datetime(
    profiles["date"].astype(str) + " 12:00:00"
)



profiles["diff_00"] = (
    profiles["profile_timestamp"] - profiles["target_00"]
).abs()

profiles["diff_12"] = (
    profiles["profile_timestamp"] - profiles["target_12"]
).abs()
profile_00 = (
    profiles.loc[
        profiles.groupby("date")["diff_00"].idxmin()
    ]
)
profile_12 = (
    profiles.loc[
        profiles.groupby("date")["diff_12"].idxmin()
    ]
)
profile_00 = profile_00.copy()
profile_00["sampling_time"] = "00:00"

profile_12 = profile_12.copy()
profile_12["sampling_time"] = "12:00"

calibration_profiles = (
    pd.concat([profile_00, profile_12])
    #   .drop_duplicates(subset="profile")
      .sort_values("profile_timestamp")
      .reset_index(drop=True)
)

In [48]:
calibration_profiles

,profile_timestamp,profile_start,profile_end,temp_2,temp_10,temp_30,temp_50,date,target_00,target_12,diff_00,diff_12,sampling_time
0,2024-04-19 15:15:57,2024-04-19 14:37:44,2024-04-19 15:54:10,3.278634,3.253439,3.258940,3.275903,2024-04-19,2024-04-19,2024-04-19 12:00:00,0 days 15:15:57,0 days 03:15:57,00:00
1,2024-04-19 15:15:57,2024-04-19 14:37:44,2024-04-19 15:54:10,3.278634,3.253439,3.258940,3.275903,2024-04-19,2024-04-19,2024-04-19 12:00:00,0 days 15:15:57,0 days 03:15:57,12:00
2,2024-04-20 00:41:20,2024-04-20 00:01:53,2024-04-20 01:20:48,3.245901,3.250998,3.266037,3.318433,2024-04-20,2024-04-20,2024-04-20 12:00:00,0 days 00:41:20,0 days 11:18:40,00:00
3,2024-04-20 12:40:17,2024-04-20 12:01:52,2024-04-20 13:18:42,3.309588,3.229459,3.306000,3.372569,2024-04-20,2024-04-20,2024-04-20 12:00:00,0 days 12:40:17,0 days 00:40:17,12:00
4,2024-04-21 00:41:18,2024-04-21 00:01:53,2024-04-21 01:20:43,3.322429,3.385510,3.398826,3.401000,2024-04-21,2024-04-21,2024-04-21 12:00:00,0 days 00:41:18,0 days 11:18:42,00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,2024-11-13 12:41:32,2024-11-13 12:01:54,2024-11-13 13:21:11,6.893120,6.897000,6.898938,NaN,2024-11-13,2024-11-13,2024-11-13 12:00:00,0 days 12:41:32,0 days 00:41:32,12:00
418,2024-11-14 00:41:42,2024-11-14 00:01:54,2024-11-14 01:21:30,6.838000,6.838118,6.841003,NaN,2024-11-14,2024-11-14,2024-11-14 12:00:00,0 days 00:41:42,0 days 11:18:18,00:00
419,2024-11-14 12:41:42,2024-11-14 12:01:54,2024-11-14 13:21:30,6.866888,6.870946,6.861921,NaN,2024-11-14,2024-11-14,2024-11-14 12:00:00,0 days 12:41:42,0 days 00:41:42,12:00
420,2024-11-15 12:41:34,2024-11-15 12:01:52,2024-11-15 13:21:16,6.845949,6.843211,6.814070,NaN,2024-11-15,2024-11-15,2024-11-15 12:00:00,0 days 12:41:34,0 days 00:41:34,00:00


In [ ]:
df_minutes = measured_df.resample('min', on='TIMESTAMP').mean().reset_index()
# Interpolate missing values using linear interpolation
df_interpolated = df_minutes.interpolate(method='linear', limit_direction='both')
df_hourly = df_interpolated.resample('h', on='TIMESTAMP').mean().reset_index()
# Get data for the year 2024
measured_2024 = df_hourly[df_hourly["TIMESTAMP"].dt.year == 2024].set_index("TIMESTAMP")

In [49]:
measured_2024

,temperature,depth
TIMESTAMP,,
2024-01-01 00:00:00,5.091867,35.252376
2024-01-01 01:00:00,5.091180,35.239331
2024-01-01 02:00:00,5.090494,35.226287
2024-01-01 03:00:00,5.089807,35.213242
2024-01-01 04:00:00,5.089120,35.200197
...,...,...
2024-12-31 19:00:00,6.941017,37.441148
2024-12-31 20:00:00,6.941149,37.429836
2024-12-31 21:00:00,6.941280,37.418525


,temperature,depth
TIMESTAMP,,
2024-01-01 00:00:00,5.091867,35.252376
2024-01-01 01:00:00,5.091180,35.239331
2024-01-01 02:00:00,5.090494,35.226287
2024-01-01 03:00:00,5.089807,35.213242
2024-01-01 04:00:00,5.089120,35.200197
...,...,...
2024-12-31 19:00:00,6.941017,37.441148
2024-12-31 20:00:00,6.941149,37.429836
2024-12-31 21:00:00,6.941280,37.418525
